# 🚀 SpaceX Falcon 9 Landing Prediction
## Notebook 1 — Data Collection

The SpaceX public REST API (`api.spacexdata.com`) is no longer available.
This notebook uses the **IBM Skills Network mirror datasets** hosted on AWS S3 —
the same data the course was originally built around, stable and always accessible.

**Goals:**
- Load the IBM-hosted SpaceX launch dataset
- Understand its structure and columns
- Handle missing values
- Save a cleaned CSV for downstream notebooks


In [14]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


### Load the IBM-hosted dataset

In [15]:
# Primary IBM dataset (Part 1 — raw launch records)
PART1_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud"
    "/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv"
)

df = pd.read_csv(PART1_URL)
print(f"Dataset shape: {df.shape}")
df.head(10)


Dataset shape: (90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


### Dataset overview

In [16]:
print("Columns:", df.columns.tolist())
print()
print(df.dtypes)


Columns: ['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude']

FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
Outcome            object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
Block             float64
ReusedCount         int64
Serial             object
Longitude         float64
Latitude          float64
dtype: object


### Missing value audit

In [17]:
null_summary = pd.DataFrame({
    "NullCount": df.isnull().sum(),
    "NullPct (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
null_summary[null_summary["NullCount"] > 0]


,NullCount,NullPct (%)
LandingPad,26,28.89


### Identify numerical vs categorical columns

In [18]:
numerical_cols   = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print("Numerical  :", numerical_cols)
print("Categorical:", categorical_cols)


Numerical  : ['FlightNumber', 'PayloadMass', 'Flights', 'Block', 'ReusedCount', 'Longitude', 'Latitude']
Categorical: ['Date', 'BoosterVersion', 'Orbit', 'LaunchSite', 'Outcome', 'LandingPad', 'Serial']


### Explore key columns

In [19]:
print("Launch Sites:")
print(df['LaunchSite'].value_counts())
print()
print("Orbits:")
print(df['Orbit'].value_counts())
print()
print("Booster Versions:")
print(df['BoosterVersion'].value_counts())


Launch Sites:
LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

Orbits:
Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
HEO       1
ES-L1     1
SO        1
GEO       1
Name: count, dtype: int64

Booster Versions:
BoosterVersion
Falcon 9    90
Name: count, dtype: int64


### Impute missing PayloadMass with column median

In [20]:
median_mass = df['PayloadMass'].median()
df['PayloadMass'].fillna(median_mass, inplace=True)
print(f"Imputed PayloadMass NaNs → median = {median_mass:.1f} kg")
print("Remaining nulls:", df['PayloadMass'].isnull().sum())


Imputed PayloadMass NaNs → median = 4701.5 kg
Remaining nulls: 0


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_44288\341684240.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['PayloadMass'].fillna(median_mass, inplace=True)


### Basic statistics

In [21]:
df.describe()


,FlightNumber,PayloadMass,Flights,Block,ReusedCount,Longitude,Latitude
count,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000
mean,45.500000,6104.959412,1.788889,3.500000,1.655556,-86.366477,29.449963
std,26.124701,4694.671720,1.213172,1.595288,1.710254,14.149518,2.141306
min,1.000000,350.000000,1.000000,1.000000,0.000000,-120.610829,28.561857
25%,23.250000,2510.750000,1.000000,2.000000,0.000000,-80.603956,28.561857
50%,45.500000,4701.500000,1.000000,4.000000,1.000000,-80.577366,28.561857
75%,67.750000,8912.750000,2.000000,5.000000,3.000000,-80.577366,28.608058
max,90.000000,15600.000000,6.000000,5.000000,5.000000,-80.577366,34.632093


### Save cleaned dataset

In [22]:
df.to_csv("../data/spacex_launches_raw.csv", index=False)
print("Saved → data/spacex_launches_raw.csv")
print(f"Shape: {df.shape}")


Saved → data/spacex_launches_raw.csv
Shape: (90, 17)
